In [112]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from pydantic import BaseModel, Field
import operator

In [113]:
load_dotenv()
model = ChatGroq(model="openai/gpt-oss-20b")

In [114]:
class EvaluationSchema(BaseModel):
    feedback: str = Field(description="Detailed feedback for essay")
    score: int = Field(description="Score out of 10", ge=0, le=10)

In [115]:
structured_model = model.with_structured_output(EvaluationSchema)

In [116]:
class EssayState(TypedDict):
    essay: str
    cot_feedback: str
    doa_feedback: str
    language_feedback: str
    individual_scores: Annotated[list[int], operator.add]
    summary_feedback: str
    average_score: float    

In [117]:
def evaluate_language(state: EssayState):
    prompt = f"Evaluate the language quality of the following essay and provide a feedback and assign a score out of 10 \n {state['essay']}"
    output = structured_model.invoke(prompt)
    print(output)
    return {"language_feedback": output.feedback, "individual_scores": [output.score] }

In [118]:
def evaluate_doa(state: EssayState):
    prompt = f'Evaluate the depth of analysis of the following essay and provide a feedback and assign a score out of 10 \n {state["essay"]}'
    output = structured_model.invoke(prompt)
    return {"doa_feedback": output.feedback, "individual_scores": [output.score]}

In [119]:
def evaluate_cot(state: EssayState):
    prompt = f'Evaluate the clarity of thought of the following essay and provide a feedback and assign a score out of 10 \n {state["essay"]}'
    output = structured_model.invoke(prompt)
    return {"cot_feedback": output.feedback, "individual_scores": [output.score]}

In [120]:
def final_evaluation(state: EssayState):
    prompt = f'Based on the following feedbacks create a summarized feedback \n language feedback - {state["language_feedback"]} \n depth of analysis feedback - {state["doa_feedback"]} \n clarity of thought feedback - {state["cot_feedback"]}'
    output = (model.invoke(prompt)).content
    average = sum(state['individual_scores']) / len(state['individual_scores'])
    return {"summary_feedback": output, "average_score": round(average, 2)}

    

In [121]:
essay = """India and AI Time

Now world change very fast because new tech call Artificial Intel… something (AI). India also want become big in this AI thing. If work hard, India can go top. But if no careful, India go back.

India have many good. We have smart student, many engine-ear, and good IT peoples. Big company like TCS, Infosys, Wipro already use AI. Government also do program “AI for All”. It want AI in farm, doctor place, school and transport.

In farm, AI help farmer know when to put seed, when rain come, how stop bug. In health, AI help doctor see sick early. In school, AI help student learn good. Government office use AI to find bad people and work fast.

But problem come also. First is many villager no have phone or internet. So AI not help them. Second, many people lose job because AI and machine do work. Poor people get more bad.

One more big problem is privacy. AI need big big data. Who take care? India still make data rule. If no strong rule, AI do bad.

India must all people together – govern, school, company and normal people. We teach AI and make sure AI not bad. Also talk to other country and learn from them.

If India use AI good way, we become strong, help poor and make better life. But if only rich use AI, and poor no get, then big bad thing happen.

So, in short, AI time in India have many hope and many danger. We must go right road. AI must help all people, not only some. Then India grow big and world say "good job India"."""

In [122]:
graph = StateGraph(EssayState)

graph.add_node('evaluate_cot', evaluate_cot)
graph.add_node('evaluate_doa', evaluate_doa)
graph.add_node('evaluate_language', evaluate_language)
graph.add_node('final_evaluation', final_evaluation)

graph.add_edge(START, 'evaluate_cot')
graph.add_edge(START, 'evaluate_doa')
graph.add_edge(START, 'evaluate_language')

graph.add_edge('evaluate_cot', 'final_evaluation')
graph.add_edge('evaluate_doa', 'final_evaluation')
graph.add_edge('evaluate_language', 'final_evaluation')

graph.add_edge('final_evaluation', END)

workflow = graph.compile()

In [123]:
initial_state = {
    'essay': essay
}
workflow.invoke(initial_state)

feedback='The essay demonstrates a basic understanding of the topic but contains numerous language errors that hinder clarity and readability. The structure is generally logical, with an introduction, body paragraphs, and a conclusion. However, many sentences suffer from grammatical mistakes, awkward phrasing, and inconsistent verb tenses. Word choice is often incorrect (e.g., "engine‑ear" instead of "engineer," "AI for All" program is not clearly described). Punctuation and capitalization errors are frequent, and some ideas are repeated without elaboration. To improve, focus on correct verb forms, use precise vocabulary, and proofread for punctuation and spelling. A more formal tone and varied sentence structures would enhance the overall quality.\n\nScore: 5/10' score=5


{'essay': 'India and AI Time\n\nNow world change very fast because new tech call Artificial Intel… something (AI). India also want become big in this AI thing. If work hard, India can go top. But if no careful, India go back.\n\nIndia have many good. We have smart student, many engine-ear, and good IT peoples. Big company like TCS, Infosys, Wipro already use AI. Government also do program “AI for All”. It want AI in farm, doctor place, school and transport.\n\nIn farm, AI help farmer know when to put seed, when rain come, how stop bug. In health, AI help doctor see sick early. In school, AI help student learn good. Government office use AI to find bad people and work fast.\n\nBut problem come also. First is many villager no have phone or internet. So AI not help them. Second, many people lose job because AI and machine do work. Poor people get more bad.\n\nOne more big problem is privacy. AI need big big data. Who take care? India still make data rule. If no strong rule, AI do bad.\n\n